# Homework 1

This homework requires you to get acquainted with the Python version of the [Polars](https://docs.pola.rs/api/python/dev/reference/index.html) DataFrame library for manipulating structured data. As you already know Pandas, you have the chance to review familiar concepts using a new (but similar) framework.

You have been provided with a text file named "sales_data.txt" that contains sales data for a company. The data includes information about sales transactions: the date, product ID, quantity sold, and revenue generated. Perform the following tasks using the Polars library:

   - Load the data from the file into a Polars DataFrame. Make sure you clean up column names and convert each column to the appropriate data type (date, string, numeric, numeric). (1 p.)
   - Remove any rows with missing values. Remove duplicate rows.
   - Calculate the price for each transaction.
   - Calculate the total revenue generated by each product and the (unweighted) average price per product.
   - Calculate the total quantity sold per date and the (unweighted) average price per date.
   - On which date did the highest revenue stream occur?
   - Identify the product category that had the smallest total quantity sold on aggregate.


In [26]:
import polars as pl

df = pl.read_csv('sales_data.txt', separator='|')

In [27]:
df.columns = [col.strip() for col in df.columns]

df = df.with_columns([
    pl.col("Date").str.strip_chars().fill_null("").replace("", None),  # Replace empty with None
    pl.col("ProductID").str.strip_chars(),
    pl.col("QuantitySold").str.strip_chars().fill_null("0").replace("", "0"),  # Replace empty with "0"
    pl.col("Revenue").str.strip_chars().fill_null("0").replace("", "0")  # Ensure empty strings are replaced
])

df = df.drop_nulls()
df = df.unique()

In [28]:
df = df.with_columns([
    pl.col("Date").str.strptime(pl.Date, strict=False),  # Convert to Date directly with strict=False
    pl.col("ProductID").cast(pl.Utf8),  # String
    pl.col("QuantitySold").cast(pl.Int32),  # Numeric
    pl.col("Revenue").cast(pl.Float64)  # Numeric
])

print(df)

shape: (11, 4)
┌────────────┬───────────┬──────────────┬─────────┐
│ Date       ┆ ProductID ┆ QuantitySold ┆ Revenue │
│ ---        ┆ ---       ┆ ---          ┆ ---     │
│ date       ┆ str       ┆ i32          ┆ f64     │
╞════════════╪═══════════╪══════════════╪═════════╡
│ 2022-01-07 ┆ A001      ┆ 12           ┆ 120.0   │
│ 2022-01-04 ┆ B002      ┆ 6            ┆ 90.3    │
│ 2022-01-05 ┆ A001      ┆ 15           ┆ 150.0   │
│ 2022-01-01 ┆ A001      ┆ 10           ┆ 100.0   │
│ 2022-01-03 ┆ A001      ┆ 5            ┆ 50.0    │
│ …          ┆ …         ┆ …            ┆ …       │
│ 2022-01-06 ┆ B002      ┆ 10           ┆ 0.0     │
│ 2022-01-06 ┆ B002      ┆ 10           ┆ 150.0   │
│ 2022-01-05 ┆ C003      ┆ 8            ┆ 120.5   │
│ 2022-01-02 ┆ B002      ┆ 8            ┆ 120.5   │
│ 2022-01-03 ┆ C003      ┆ 12           ┆ 180.75  │
└────────────┴───────────┴──────────────┴─────────┘


In [29]:
df = df.with_columns((pl.col("Revenue") / pl.col("QuantitySold")).alias("Price"))

In [30]:
total_revenue_by_product = df.group_by("ProductID").agg([
    pl.col("Revenue").sum().alias("TotalRevenue"),
    (pl.col("Revenue") / pl.col("QuantitySold")).mean().alias("AveragePrice")  # Calculate average price
])

print(total_revenue_by_product)

shape: (3, 3)
┌───────────┬──────────────┬──────────────┐
│ ProductID ┆ TotalRevenue ┆ AveragePrice │
│ ---       ┆ ---          ┆ ---          │
│ str       ┆ f64          ┆ f64          │
╞═══════════╪══════════════╪══════════════╡
│ B002      ┆ 360.8        ┆ 11.278125    │
│ A001      ┆ 420.0        ┆ 10.0         │
│ C003      ┆ 391.25       ┆ 15.041667    │
└───────────┴──────────────┴──────────────┘


In [31]:
total_quantity_per_date = df.group_by("Date").agg([
    pl.col("QuantitySold").sum().alias("TotalQuantitySold"),
    (pl.col("Revenue") / pl.col("QuantitySold")).mean().alias("AveragePrice")  # Calculate average price
])

print (total_quantity_per_date)

shape: (7, 3)
┌────────────┬───────────────────┬──────────────┐
│ Date       ┆ TotalQuantitySold ┆ AveragePrice │
│ ---        ┆ ---               ┆ ---          │
│ date       ┆ i32               ┆ f64          │
╞════════════╪═══════════════════╪══════════════╡
│ 2022-01-06 ┆ 20                ┆ 7.5          │
│ 2022-01-07 ┆ 18                ┆ 12.5         │
│ 2022-01-02 ┆ 8                 ┆ 15.0625      │
│ 2022-01-04 ┆ 6                 ┆ 15.05        │
│ 2022-01-03 ┆ 17                ┆ 12.53125     │
│ 2022-01-01 ┆ 10                ┆ 10.0         │
│ 2022-01-05 ┆ 23                ┆ 12.53125     │
└────────────┴───────────────────┴──────────────┘


In [36]:
highest_revenue_date = df.group_by("Date").agg([
    pl.col("Revenue").sum().alias("TotalRevenue")
]).sort("TotalRevenue").reverse()


top_date_revenue = highest_revenue_date[0]


print(top_date_revenue)

print(highest_revenue_date)

shape: (1, 2)
┌────────────┬──────────────┐
│ Date       ┆ TotalRevenue │
│ ---        ┆ ---          │
│ date       ┆ f64          │
╞════════════╪══════════════╡
│ 2022-01-05 ┆ 270.5        │
└────────────┴──────────────┘
shape: (7, 2)
┌────────────┬──────────────┐
│ Date       ┆ TotalRevenue │
│ ---        ┆ ---          │
│ date       ┆ f64          │
╞════════════╪══════════════╡
│ 2022-01-05 ┆ 270.5        │
│ 2022-01-03 ┆ 230.75       │
│ 2022-01-07 ┆ 210.0        │
│ 2022-01-06 ┆ 150.0        │
│ 2022-01-02 ┆ 120.5        │
│ 2022-01-01 ┆ 100.0        │
│ 2022-01-04 ┆ 90.3         │
└────────────┴──────────────┘


In [37]:
smallest_quantity_product = df.group_by("ProductID").agg([
    pl.col("QuantitySold").sum().alias("TotalQuantitySold")
]).sort("TotalQuantitySold")

least_product_quantity = smallest_quantity_product[0]

print(least_product_quantity)
print(smallest_quantity_product)

shape: (1, 2)
┌───────────┬───────────────────┐
│ ProductID ┆ TotalQuantitySold │
│ ---       ┆ ---               │
│ str       ┆ i32               │
╞═══════════╪═══════════════════╡
│ C003      ┆ 26                │
└───────────┴───────────────────┘
shape: (3, 2)
┌───────────┬───────────────────┐
│ ProductID ┆ TotalQuantitySold │
│ ---       ┆ ---               │
│ str       ┆ i32               │
╞═══════════╪═══════════════════╡
│ C003      ┆ 26                │
│ B002      ┆ 34                │
│ A001      ┆ 42                │
└───────────┴───────────────────┘
